In [21]:
import cv2 as cv
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.cluster import DBSCAN
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as transforms


In [22]:
def preprocess_image(img : np.ndarray) -> np.ndarray:    
    _, thresh = cv.threshold(
        cv.Canny(
        cv.GaussianBlur(
        cv.cvtColor(
        img, 
        cv.COLOR_BGR2GRAY), 
        (1,1), 1),
        130, 255),
        130, 255, cv.THRESH_BINARY)
    return thresh

In [23]:
def split_into_loaders(X: np.array, y: np.array):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    return train_loader, test_loader

In [24]:
class BrailleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv2d(1, 32, 3),  # input channels = 1
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 128),  # 28x28 -> conv & pool -> 5x5
            nn.ReLU(),
            nn.Linear(128, 26)  # 26 output classes
        )

    def forward(self, x):
        x = self.conv_layer(x)
        x = self.fc_layer(x)
        return x

In [25]:
def train_model(train_loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = BrailleCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    EPOCHS = 32
    for epoch in range(EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/total:.4f} | Accuracy: {100 * correct / total:.2f}%")

    return model, device

In [26]:
def evaluate_model(test_loader, model, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    print(f"Test Accuracy: {100 * correct / total:.2f}%")

    torch.save(model.state_dict(), "braille_cnn_model.pth")
    print("Model saved successfully!")


In [27]:
def merge_close_boxes(boxes):
    """
    Merge nearby/overlapping bounding boxes into larger boxes (Braille cells).
    max_distance controls how close boxes need to be to merge.
    """
    merged = []
    while boxes:
        x, y, w, h = boxes.pop(0)
        box1 = [x, y, x + w, y + h]

        to_merge = []
        for b in boxes:
            x2, y2, w2, h2 = b
            box2 = [x2, y2, x2 + w2, y2 + h2]

            # Check if boxes overlap or are close enough to be in the same Braille cell
            if not (box1[2] + 1*w < box2[0] or
                    box1[0] - 0.75*w> box2[2] or
                    box1[3] + 3*h < box2[1] or
                    box1[1] > box2[3]):
                to_merge.append(b)

        # Merge all nearby boxes
        for b in to_merge:
            boxes.remove(b)
            x2, y2, w2, h2 = b
            box1[0] = min(box1[0], x2)
            box1[1] = min(box1[1], y2)
            box1[2] = max(box1[2], x2 + w2)
            box1[3] = max(box1[3], y2 + h2)

        merged.append((box1[0], box1[1], box1[2] - box1[0], box1[3] - box1[1]))

    return merged

In [28]:
def merge_overlapping_boxes(boxes):
    """
    Merge nearby/overlapping bounding boxes into larger boxes (Braille cells).
    max_distance controls how close boxes need to be to merge.
    """
    merged = []
    while boxes:
        x, y, w, h = boxes.pop(0)
        box1 = [x, y, x + w, y + h]

        to_merge = []
        for b in boxes:
            x2, y2, w2, h2 = b
            box2 = [x2, y2, x2 + w2, y2 + h2]

            # Check if boxes overlap or are close enough to be in the same Braille cell
            if not (box1[2] < box2[0] or
                    box1[0] > box2[2] or
                    box1[3] < box2[1] or
                    box1[1] > box2[3]):
                to_merge.append(b)

        # Merge all nearby boxes
        for b in to_merge:
            boxes.remove(b)
            x2, y2, w2, h2 = b
            box1[0] = min(box1[0], x2)
            box1[1] = min(box1[1], y2)
            box1[2] = max(box1[2], x2 + w2)
            box1[3] = max(box1[3], y2 + h2)

        merged.append((box1[0], box1[1], box1[2] - box1[0], box1[3] - box1[1]))

    return merged

In [29]:
def sort_boxes_rowwise_with_tolerance(boxes, y_tolerance=10):
    # Step 1: Sort all boxes by Y (top) coordinate
    boxes_sorted = sorted(boxes, key=lambda b: b[1])
    
    rows = []
    current_row = []
    
    for box in boxes_sorted:
        x, y, w, h = box
        if not current_row:
            current_row.append(box)
        else:
            # Compare Y with the first box in the current row
            _, ref_y, _, ref_h = current_row[0]
            if abs(y - ref_y) <= y_tolerance:
                current_row.append(box)
            else:
                # Sort current row by X (left to right)
                current_row = sorted(current_row, key=lambda b: b[0])
                rows.append(current_row)
                current_row = [box]
    
    # Sort last row
    if current_row:
        current_row = sorted(current_row, key=lambda b: b[0])
        rows.append(current_row)

    # Flatten the list of rows
    return [box for row in rows for box in row]

In [30]:
def find_dot_centers(thresh_img: np.ndarray) -> list:
    contours, _ = cv.findContours(thresh_img, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)
    # centers = []

    boxes = [cv.boundingRect(cnt) for cnt in contours if 8 < cv.contourArea(cnt) < 100]
    boxes_sorted = sort_boxes_rowwise_with_tolerance(boxes, y_tolerance=12)
    # boxes_sorted = sorted(boxes, key=lambda b: (b[1], b[0]))

    boxes_2 = merge_close_boxes(boxes_sorted)
    
    boxes_3 = sort_boxes_rowwise_with_tolerance(boxes_2, y_tolerance=5)
    boxes_4 = merge_overlapping_boxes(boxes_3)

    return boxes_4

In [31]:
def extract_cells(thresh_img: np.ndarray, cell_groups: list) -> tuple:
    """
    Given a thresholded image and list of bounding boxes (cell_groups),
    extract 2x3 grid of Braille dot images from each cell.

    Returns:
        - List of 2x3 numpy arrays (each array contains 6 preprocessed dot images).
        - List of corresponding bounding boxes for reference/debugging.
    """
    processed_cells = []
    boxes = []


    for (x, y, w, h) in cell_groups:
        pad = 5
        x1, y1 = max(0, x - pad), max(0, y - pad)
        x2, y2 = min(thresh_img.shape[1], x + w + pad), min(thresh_img.shape[0], y + h + pad)
        cell_img = thresh_img[y1:y2, x1:x2]
        boxes.append((x1, y1, x2, y2))

        # Divide cell image into 2x3 grid (cols x rows = 2 x 3 = 6 parts)
        grid_rows, grid_cols = 3, 2
        cell_height, cell_width = cell_img.shape
        row_height = cell_height // grid_rows
        col_width = cell_width // grid_cols

        grid = []
        for row in range(grid_rows):
            for col in range(grid_cols):
                x_start = col * col_width
                y_start = row * row_height
                x_end = (col + 1) * col_width
                y_end = (row + 1) * row_height

                dot_img = cell_img[y_start:y_end, x_start:x_end]
                dot_img = cv.resize(dot_img, (28, 28))  # or 32x32 based on your model input
                dot_img = dot_img.astype(np.float32) / 255.0  # normalize
                grid.append(dot_img)

        grid_array = np.array(grid).reshape(3, 2, 28, 28)  # shape: 3 rows × 2 cols × H × W
        processed_cells.append(grid_array)

    return processed_cells, boxes


In [32]:
def predict_braille_text_from_image(img_path, model, device):
    img = cv.imread(str(img_path))
    thresh = preprocess_image(img)
    cell_groups = find_dot_centers(thresh)
    cells, _ = extract_cells(thresh, cell_groups)

    predicted_text = ''
    model.eval()

    transform = transforms.Compose([
        transforms.ToTensor(),  # Converts HWC to CHW and scales 0–255 to 0–1
        transforms.Resize((28, 28)),  # Ensures input size
    ])

    for i, cell in enumerate(cells):
        # Visualize the original cell
        # print(f"Cell #{i + 1}: shape={cell.shape}")
        # cv.imshow("Braille Cell", cell)
        # cv.waitKey(0)
        # cv.destroyAllWindows()

        h, w = cell.shape
        cell_imgs = []
        dot_rows, dot_cols = 3, 2
        dot_h, dot_w = h // dot_rows, w // dot_cols

        for row in range(dot_rows):
            for col in range(dot_cols):
                y1, y2 = row * dot_h, (row + 1) * dot_h
                x1, x2 = col * dot_w, (col + 1) * dot_w
                dot_img = cell[y1:y2, x1:x2]
                dot_img = cv.resize(dot_img, (28, 28), interpolation=cv.INTER_AREA)
                dot_tensor = transform(dot_img)  # shape: [1, 28, 28]
                cell_imgs.append(dot_tensor)

        # Create tensor of shape [6, 1, 28, 28]
        cell_tensor = torch.stack(cell_imgs).to(device)

        with torch.no_grad():
            output = model(cell_tensor)  # shape: [6, num_classes]
            pred_classes = torch.argmax(output, dim=1)  # [6]

        # Decode predictions — e.g., using class to letter map
        for pred in pred_classes:
            predicted_text += chr(97 + pred.item())  # assuming 0=a, 1=b, etc.

    return predicted_text


In [33]:
def get_actual_text_from_yolo_labels(label_path):
    with open(label_path, "r", encoding="utf-8") as f:
        yolo_lines = f.readlines()

    labels = []
    for line in yolo_lines:
        parts = line.strip().split()
        cls_id = int(parts[0])
        x_center = float(parts[1])
        y_center = float(parts[2])
        labels.append((cls_id, x_center, y_center))

    actual_text = ''.join(chr(97 + cls_id) for cls_id, _, _ in labels)
    return actual_text

In [34]:
def compare_predicted_and_actual(predicted_text, actual_text):
    min_len = min(len(predicted_text), len(actual_text))
    matches = sum(predicted_text[i] == actual_text[i] for i in range(min_len))
    accuracy = matches / max(len(actual_text), 1) * 100  # avoid divide-by-zero
    print(f"\nActual Text:    {actual_text}")
    print(f"Predicted Text: {predicted_text}")
    print(f"Character-level Accuracy: {accuracy:.2f}%")
    return accuracy

In [35]:
def segment_braille_dots(thresh_img: np.ndarray) -> list:
    contours, _ = cv.findContours(thresh_img, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    dot_contours = []
    for cnt in contours:
        area = cv.contourArea(cnt)
        if 10 < area < 100:
            dot_contours.append(cnt)
    return dot_contours

In [36]:
def find_dot_centers(thresh_img: np.ndarray) -> list:
    contours, _ = cv.findContours(thresh_img, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)
    # centers = []

    boxes = [cv.boundingRect(cnt) for cnt in contours if 8 < cv.contourArea(cnt) < 100]
    boxes_sorted = sort_boxes_rowwise_with_tolerance(boxes, y_tolerance=12)
    # boxes_sorted = sorted(boxes, key=lambda b: (b[1], b[0]))

    boxes_2 = merge_close_boxes(boxes_sorted)
    
    boxes_3 = sort_boxes_rowwise_with_tolerance(boxes_2, y_tolerance=5)
    boxes_4 = merge_overlapping_boxes(boxes_3)

    return boxes_4

In [37]:
DATASET_DIR = r"C:\Users\veerk\OneDrive\Desktop\Braille_To_Speech\Braille Dataset\Braille Dataset"

In [38]:
X, y = [], []
label_map = {chr(97 + i): i for i in range(26)} 

for img_path in Path(DATASET_DIR).iterdir():
    processed = preprocess_image(cv.imread(img_path))
    X.append(processed)
    y.append(label_map[img_path.name[0]])

X = np.array(X).astype(np.float32).reshape(-1, 1, 28, 28) / 255.0 
y = np.array(y).astype(np.int64)

train_loader, test_loader = split_into_loaders(X, y)
model, device = train_model(train_loader)

evaluate_model(test_loader, model, device)

Epoch 1/32 | Loss: 3.0372 | Accuracy: 9.62%
Epoch 2/32 | Loss: 1.9447 | Accuracy: 51.04%
Epoch 3/32 | Loss: 1.4561 | Accuracy: 62.98%
Epoch 4/32 | Loss: 1.2289 | Accuracy: 65.30%
Epoch 5/32 | Loss: 1.0294 | Accuracy: 71.15%
Epoch 6/32 | Loss: 0.9049 | Accuracy: 73.64%
Epoch 7/32 | Loss: 0.7568 | Accuracy: 78.69%
Epoch 8/32 | Loss: 0.6369 | Accuracy: 81.81%
Epoch 9/32 | Loss: 0.5516 | Accuracy: 83.57%
Epoch 10/32 | Loss: 0.4612 | Accuracy: 86.86%
Epoch 11/32 | Loss: 0.3854 | Accuracy: 88.86%
Epoch 12/32 | Loss: 0.3160 | Accuracy: 90.54%
Epoch 13/32 | Loss: 0.2590 | Accuracy: 92.39%
Epoch 14/32 | Loss: 0.1835 | Accuracy: 94.95%
Epoch 15/32 | Loss: 0.2037 | Accuracy: 93.75%
Epoch 16/32 | Loss: 0.1520 | Accuracy: 95.03%
Epoch 17/32 | Loss: 0.1224 | Accuracy: 96.79%
Epoch 18/32 | Loss: 0.1247 | Accuracy: 96.55%
Epoch 19/32 | Loss: 0.0649 | Accuracy: 98.32%
Epoch 20/32 | Loss: 0.1070 | Accuracy: 96.79%
Epoch 21/32 | Loss: 0.0644 | Accuracy: 98.00%
Epoch 22/32 | Loss: 0.0664 | Accuracy: 97.92

In [39]:
image_path = r"C:\Users\veerk\OneDrive\Desktop\Braille_To_Speech\Braille Dataset\Braille Document\datasets-braille\data\images\test\IMG_4479_jpg.rf.74bfb6d141e15e92aa9aa0cb70236cd5.jpg"
label_path = r"C:\Users\veerk\OneDrive\Desktop\Braille_To_Speech\Braille Dataset\Braille Document\datasets-braille\data\labels\test\IMG_4479_jpg.rf.74bfb6d141e15e92aa9aa0cb70236cd5.txt"

predicted = predict_braille_text_from_image(image_path, model, device)
actual = get_actual_text_from_yolo_labels(label_path)
compare_predicted_and_actual(predicted, actual)   

ValueError: too many values to unpack (expected 2)